# Part 4 — Vector Databases & Indexing

*From exact, brute-force k-NN to an approximate index, and the one trade-off the whole field turns on: speed versus recall.*

📖 Read the essay: https://www.mefby.com/essays/vector-databases  
🐍 Companion script: `vector_db.py`

**What you'll build**

- A seeded NumPy point cloud standing in for chunk embeddings (no model, no network).
- `brute_force_topk` — the exact, O(n) baseline from Part 3, named and measured.
- `recall@k` — the word for the sliver of accuracy that approximate search gives up.
- `kmeans` in plain NumPy — the clustering step behind IVF.
- `IVFIndex` — k-means shelves plus an `nprobe` dial, the speed-vs-recall curve as runnable code.
- A `nprobe` sweep that prints recall and speedup side by side.
- A `Product Quantization` sketch — compressing the vectors themselves.

> This notebook runs **fully offline**: numpy only, nothing to download. The real production path (FAISS, the reference ANN toolkit) is shown as the intended code and used automatically if it happens to be installed; otherwise a transparent, dependency-free NumPy stand-in keeps every cell executable.

## Setup

Everything in this part is pure NumPy and the standard library. No embedding model, no API key, no network. We pull in `numpy` for the vector math and `time` so we can clock the brute-force baseline.

In [ ]:
import time

import numpy as np

# A fixed seed everywhere -> identical numbers every run, on any machine.
print('numpy', np.__version__, '— offline, deterministic, no model required.')

## A point cloud standing in for chunk embeddings

In a real RAG app, these vectors would come from an embedding model (Part 2), already unit-normalized so a dot product reads straight off as cosine similarity (Part 3). Here we *synthesize* them so the notebook runs with no model and no network: same seed in, identical numbers out.

One deliberate choice: we draw the points in **clumps**, not uniform noise. Real embeddings cluster by meaning — refund chunks sit near refund chunks — and that clustering is exactly what IVF will exploit later. Each point is a random blob center plus a little Gaussian jitter.

In [ ]:
def unit_normalize(x):
    """Scale each row to length 1 so dot product == cosine similarity (Part 3)."""
    norms = np.linalg.norm(x, axis=-1, keepdims=True)
    return x / np.maximum(norms, 1e-12)          # guard against a zero vector


def make_point_cloud(n=2000, d=32, n_blobs=12, seed=42):
    """Return an (n, d) matrix of unit vectors arranged in n_blobs clusters."""
    rng = np.random.default_rng(seed)            # seeded: reproducible, not random noise
    centers = rng.normal(size=(n_blobs, d))      # one random center per blob
    labels = rng.integers(0, n_blobs, size=n)    # assign each point to a blob
    # Each point = its blob center + a little Gaussian jitter.
    vectors = centers[labels] + 0.35 * rng.normal(size=(n, d))
    return unit_normalize(vectors), rng

In [ ]:
N, D, K = 2000, 32, 10
vectors, rng = make_point_cloud(n=N, d=D, n_blobs=12, seed=42)
print(f'point cloud: N={N} vectors, d={D} dims, drawn in 12 clumps (seeded)')
print('every row is unit length ->', np.allclose(np.linalg.norm(vectors, axis=1), 1.0))

## The baseline we just outgrew: brute-force k-NN

Let me name what we have been doing. To answer a query we compare its vector against the stored vector of **every** chunk, score each one, and keep the closest `k`. Comparing against the whole collection and returning the genuine `k` nearest is **exact nearest-neighbor search**, or, because you do it by sheer force of checking everything, **brute-force k-NN**.

It has one wonderful property and one fatal one. Wonderful: it is *exactly right* — it inspects every candidate, so the `k` it returns really are the closest. Fatal: it is **O(n)** — one dot product per stored vector, so doubling the collection doubles the work. The code is almost embarrassingly short.

In [ ]:
def brute_force_topk(query, vectors, k=4):
    """Return the indices of the k vectors most similar to `query` (exact)."""
    scores = vectors @ query                     # one dot product per vector -> O(N)
    return np.argsort(-scores)[:k]               # indices of the k highest scores

In [ ]:
# A query that sits near vector 0: nudge it a little so it has a real nearby answer.
query = unit_normalize(vectors[0] + 0.1 * rng.normal(size=D))

t0 = time.perf_counter()
exact_top = brute_force_topk(query, vectors, k=K)
dt = (time.perf_counter() - t0) * 1e3

print(f'exact top-{K} indices : {[int(i) for i in exact_top]}')
print(f'comparisons         : {N} (one dot product per stored vector)')
print(f'time                : {dt:.3f} ms')

For a few thousand chunks this is genuinely all you need — `vectors @ query` is the exact operation modern hardware is most optimized for. The trouble starts when `N` stops being thousands. A serious knowledge base is millions or billions of vectors, and O(n) per query, times many queries per second, is a wall. We need to find the nearest vectors *without looking at all of them*.

## recall@k: the word for the sliver we give up

The pivot the whole field turns on is a genuine change of goal: we stop insisting on the *exact* nearest neighbors and accept *almost always the right ones*. That single concession buys speedups of orders of magnitude — the foundation of **Approximate Nearest Neighbor (ANN)** search.

To talk about the sliver we give up, we need a word for it. **Recall**, in this nearest-neighbor sense, is the fraction of the true top-k that an approximate search actually returns. If the exact top-10 has ten specific chunks and ANN returns eight of them, that is **0.8** recall. It is a *set* overlap — the order within the `k` does not matter.

In [ ]:
def recall_at_k(approx_idx, true_idx):
    """Fraction of the true top-k that the approximate result recovered."""
    true_set = set(int(i) for i in true_idx)
    if not true_set:
        return 1.0
    found = sum(1 for i in approx_idx if int(i) in true_set)
    return found / len(true_set)

In [ ]:
# Two sanity checks straight from the essay.
perfect = recall_at_k(exact_top, exact_top)            # the set against itself -> 1.0
eight_of_ten = recall_at_k(exact_top[:8], exact_top)   # found only 8 of the true 10
print(f'recall of the exact set against itself : {perfect:.2f}   (must be 1.00)')
print(f'recall when only 8 of the true 10 found: {eight_of_ten:.2f}   (the 0.8 / 80% example)')

## k-means: the clustering step behind IVF

**IVF** (the index we build next) groups stored vectors into clusters *by proximity* — the essay's library shelves. **k-means** is the algorithm that does the grouping: pick `k` centroids, assign every point to its nearest centroid, move each centroid to the mean of its members, and repeat until nothing moves. A **centroid** is just the average position of a cluster's members — its representative address.

We write it in plain NumPy so nothing is hidden. The assignment step uses the expanded squared-Euclidean distance `||a-b||^2 = ||a||^2 - 2 a·b + ||b||^2`, then `argmin` over centroids picks each point's cluster.

In [ ]:
def kmeans(vectors, n_clusters, n_iter=25, seed=0):
    """Lloyd's algorithm. Returns (centroids, assignment-per-vector)."""
    rng = np.random.default_rng(seed)
    n = vectors.shape[0]
    # Initialize centroids at n_clusters distinct randomly chosen points.
    init = rng.choice(n, size=n_clusters, replace=False)
    centroids = vectors[init].copy()
    assign = np.zeros(n, dtype=int)
    for _ in range(n_iter):
        # Assign: each point to the nearest centroid by squared Euclidean dist.
        dists = (
            np.sum(vectors ** 2, axis=1, keepdims=True)
            - 2.0 * vectors @ centroids.T
            + np.sum(centroids ** 2, axis=1)[None, :]
        )
        new_assign = np.argmin(dists, axis=1)
        if np.array_equal(new_assign, assign):
            break                                # converged: nothing moved
        assign = new_assign
        # Update: move each centroid to the mean of its members.
        for c in range(n_clusters):
            members = vectors[assign == c]
            if len(members) > 0:
                centroids[c] = members.mean(axis=0)
            else:
                centroids[c] = vectors[rng.integers(0, n)]  # re-seed an empty cluster
    return centroids, assign

In [ ]:
# A quick look: cluster the cloud into a handful of shelves and inspect the split.
demo_centroids, demo_assign = kmeans(vectors, n_clusters=12, seed=0)
demo_sizes = np.bincount(demo_assign, minlength=12)
print(f'12 centroids learned, each of shape {demo_centroids.shape[1:]}')
print(f'members per cluster: {demo_sizes.tolist()}')
print(f'total members = {int(demo_sizes.sum())} (every one of the {N} vectors lands on a shelf)')

## IVF: sorting vectors onto shelves

**IVF**, the **Inverted File Index**, is about not searching shelves you already know are irrelevant. Picture a library: books are grouped by subject, so to find Roman history you walk straight to the history section instead of scanning every shelf.

**Build time:** cluster the vectors with k-means; each centroid is a shelf label, and an *inverted list* records which vectors live on which shelf. **Query time:** compare the query against the (few) centroids, pick the `nprobe` nearest shelves, and search **only inside those**.

The essay's worked example: a million vectors split into a thousand clusters of a thousand each, looking in one cluster, replaces a million comparisons with about 1000 (centroids) + 1000 (one cluster) ≈ 2000. `nprobe` is the dial that slides us along the speed-recall curve.

Here is the whole index in one class. `build` runs k-means and records the inverted lists. `search` is the heart of IVF, and notice its two-stage cost: one comparison per centroid (cheap — there are few of them), then one comparison per candidate inside the opened shelves. We track `last_comparisons` so the sweep below can read off exactly how much work each query did. After defining it we build the index over our cloud and look at the shelves it created.

In [ ]:
class IVFIndex:
    def __init__(self, n_clusters=40, seed=0):
        self.n_clusters = n_clusters
        self.seed = seed
        self.centroids = None
        self.inverted_lists = None               # cluster id -> array of vector ids
        self.vectors = None
        self.last_comparisons = 0                # comparisons made by the last search

    def build(self, vectors):
        """Train the centroids and bucket every vector onto its shelf."""
        self.vectors = vectors
        self.centroids, assign = kmeans(vectors, self.n_clusters, seed=self.seed)
        # Inverted lists: the 'file' that maps each shelf to its members.
        self.inverted_lists = [
            np.where(assign == c)[0] for c in range(self.n_clusters)
        ]
        return self

    def search(self, query, k=4, nprobe=1):
        """Approximate top-k: search only the nprobe nearest clusters."""
        # 1) Compare the query against the centroids only (cheap: few of them).
        centroid_scores = self.centroids @ query           # the nprobe stage
        comparisons = self.n_clusters                       # one per centroid
        nearest_clusters = np.argsort(-centroid_scores)[:nprobe]

        # 2) Gather the members of those clusters -- the only vectors we score.
        candidate_ids = np.concatenate(
            [self.inverted_lists[c] for c in nearest_clusters]
        ) if nprobe > 0 else np.array([], dtype=int)

        if candidate_ids.size == 0:
            self.last_comparisons = comparisons
            return np.array([], dtype=int)

        # 3) Brute force, but only WITHIN the opened shelves (the whole point).
        scores = self.vectors[candidate_ids] @ query
        comparisons += candidate_ids.size                   # one per candidate
        self.last_comparisons = comparisons

        top_local = np.argsort(-scores)[:k]
        return candidate_ids[top_local]

In [ ]:
n_clusters = 40
index = IVFIndex(n_clusters=n_clusters, seed=0).build(vectors)
sizes = [len(lst) for lst in index.inverted_lists]
print(f'clusters      : {n_clusters}')
print(f'cluster sizes : min={min(sizes)}, max={max(sizes)}, mean={np.mean(sizes):.1f}')

# Quick check: at a generous nprobe the index should recover the exact answer.
approx_top = index.search(query, k=K, nprobe=8)
print(f'\nIVF top-{K} (nprobe=8): {[int(i) for i in approx_top]}')
print(f'comparisons         : {index.last_comparisons} (vs {N} for brute force)')
print(f'recall@{K}           : {recall_at_k(approx_top, exact_top):.2f}')

## The real production path: FAISS (shown, optional)

Our `IVFIndex` is a teaching re-implementation of exactly what production toolkits do. The reference one is **FAISS** — the *library / raw index* row in the essay's landscape table: not a database, but the canonical toolkit for the ANN indexes themselves. Below is the intended real code, an `IndexIVFFlat` with the same `nprobe` dial and the same meaning.

We wrap the import in `try/except` so it never blocks the notebook. If FAISS is installed it gets built; if not, we print a clear label and fall straight back to the transparent NumPy `IVFIndex` above. This is the same guarded pattern Part 2 uses for the embedding model — the real library is the headline, the stand-in keeps the cell runnable offline.

In [ ]:
def build_faiss_ivf(vectors, n_clusters=40, nprobe=1):
    """Build a real FAISS IndexIVFFlat. Returns None if faiss isn't installed."""
    try:
        import faiss
    except ImportError:
        return None                              # transparent fallback: caller uses IVFIndex
    d = vectors.shape[1]
    quantizer = faiss.IndexFlatIP(d)             # inner product == cosine (unit vectors)
    index = faiss.IndexIVFFlat(quantizer, d, n_clusters, faiss.METRIC_INNER_PRODUCT)
    vectors = np.ascontiguousarray(vectors, dtype='float32')
    index.train(vectors)                         # k-means under the hood
    index.add(vectors)
    index.nprobe = nprobe                        # the same dial, same meaning
    return index


faiss_index = build_faiss_ivf(vectors, n_clusters=n_clusters, nprobe=1)
if faiss_index is not None:
    print('FAISS available -> real IndexIVFFlat built (production path).')
else:
    print('[FAISS unavailable] -> using the transparent NumPy IVFIndex above.')
    print('Same algorithm, same nprobe dial; the numbers below come from our code.')

## The nprobe sweep: the speed-vs-recall curve, in numbers

Now we can *watch* the trade-off. For each `nprobe` we run many queries, average the recall against the exact top-k, and count the comparisons (our proxy for work, i.e. speed).

- **Low nprobe:** a tiny comparison count, lower recall — we miss neighbors sitting just across a cluster boundary we never opened.
- **High nprobe:** recall climbs toward 1.0 as comparisons climb toward brute force.

We compute the exact answers once and reuse them as ground truth for every setting.

In [ ]:
def nprobe_sweep(index, queries, vectors, k=10, nprobe_values=(1, 2, 4, 8, 16)):
    """Return (rows, brute_comparisons): one row per nprobe with mean recall + work."""
    # Exact answers once, reused as ground truth for every nprobe setting.
    exact = [brute_force_topk(q, vectors, k=k) for q in queries]
    brute_comparisons = vectors.shape[0]         # brute force compares against all N

    rows = []
    for nprobe in nprobe_values:
        recalls, comps = [], []
        for q, true_idx in zip(queries, exact):
            approx = index.search(q, k=k, nprobe=nprobe)
            recalls.append(recall_at_k(approx, true_idx))
            comps.append(index.last_comparisons)
        rows.append({
            'nprobe': nprobe,
            'recall': float(np.mean(recalls)),
            'comparisons': float(np.mean(comps)),
            'speedup': brute_comparisons / float(np.mean(comps)),
        })
    return rows, brute_comparisons

In [ ]:
# Queries = stored vectors nudged a little, so each has a real nearby answer.
n_queries = 200
q_ids = rng.choice(N, size=n_queries, replace=False)
queries = unit_normalize(vectors[q_ids] + 0.1 * rng.normal(size=(n_queries, D)))

rows, brute_comps = nprobe_sweep(index, queries, vectors, k=K,
                                 nprobe_values=(1, 2, 4, 8, 16))
print(f'brute force compares against all {int(brute_comps)} vectors per query.\n')
print(f"{'nprobe':>6} | {'recall@%d' % K:>9} | {'comparisons':>12} | {'speedup':>8}")
print(f"{'-'*6}-+-{'-'*9}-+-{'-'*12}-+-{'-'*8}")
for r in rows:
    print(f"{r['nprobe']:>6} | {r['recall']:>9.3f} | {r['comparisons']:>12.0f} | {r['speedup']:>7.1f}x")

Read it exactly as the essay does: turn `nprobe` up and recall rises, but you pay in comparisons (less speed); turn it down and you fly, but miss more of the true neighbors. **`nprobe` is the dial on the curve.** There is no free lunch, only a choice of where on the speed-recall curve to live.

## Product Quantization: shrinking the vectors themselves

There is a third lever, orthogonal to the index: **compression**. A million 768-dimensional float32 vectors is several gigabytes *before* any index. **Product Quantization (PQ)** is the name to recognize.

The intuition: instead of storing every vector at full precision, PQ chops each vector into `m` short **sub-vectors** and replaces each piece with the nearest entry from a small learned **codebook** (one codebook per sub-space, learned by k-means — which is why we already have it). A long list of precise floats collapses into `m` tiny integer codes. Distances can be computed on the codes; the price is a little lost precision (recall). Paired with clustering it becomes **IVF+PQ**: cluster to narrow the search, then store the members compressed so far more fit in memory.

In [ ]:
def product_quantize(vectors, m=4, ksub=16, seed=0):
    """Train m codebooks of ksub entries each; encode vectors into m codes apiece.

    Returns (codes, codebooks):
      codes     : (n, m) int array -- each vector is now just m small indices
      codebooks : list of m arrays, each (ksub, d/m) -- the learned sub-vectors
    """
    n, d = vectors.shape
    assert d % m == 0, 'sub-vector count m must divide the dimension d'
    dsub = d // m                                 # length of each sub-vector
    codebooks, codes = [], np.zeros((n, m), dtype=int)
    for j in range(m):
        sub = vectors[:, j * dsub:(j + 1) * dsub]  # the j-th slice of every vector
        # Learn this sub-space's codebook by k-means; centroids ARE the codebook.
        book, assign = kmeans(sub, ksub, seed=seed + j)
        codebooks.append(book)
        codes[:, j] = assign                       # store the nearest code, not the floats
    return codes, codebooks


def pq_reconstruct(codes, codebooks):
    """Rebuild approximate vectors by stitching each sub-vector's codebook entry."""
    pieces = [codebooks[j][codes[:, j]] for j in range(len(codebooks))]
    return np.concatenate(pieces, axis=1)


def pq_memory_bytes(n, d, m, ksub):
    """Compare raw float32 storage against PQ codes (codebooks are tiny, shared)."""
    raw = n * d * 4                                # 4 bytes per float32
    code_bytes = 1 if ksub <= 256 else 2          # one byte per code if <=256 entries
    compressed = n * m * code_bytes               # the codebooks are negligible at scale
    return raw, compressed

In [ ]:
m, ksub = 4, 16                               # 4 sub-vectors, 16-entry codebooks
codes, codebooks = product_quantize(vectors, m=m, ksub=ksub, seed=0)
approx = pq_reconstruct(codes, codebooks)

# How faithful is the compressed form? Mean cosine between original and rebuilt.
fidelity = float(np.mean(np.sum(unit_normalize(approx) * vectors, axis=1)))
raw_bytes, pq_bytes = pq_memory_bytes(N, D, m, ksub)

print(f'each {D}-d vector -> {m} codes from {ksub}-entry codebooks')
print(f'one vector now    : {codes[0].tolist()}  (was {D} floats)')
print(f'memory raw float32: {raw_bytes:,} bytes')
print(f'memory PQ codes   : {pq_bytes:,} bytes  ({raw_bytes / pq_bytes:.0f}x smaller)')
print(f'rebuild fidelity  : {fidelity:.3f} mean cosine to the original')
print('(smaller memory, a small precision/recall cost -- the PQ bargain.)')

## The headline demo, end to end

Finally, the same progression the companion `vector_db.py` prints when you run it: brute force (exact, O(n)) → IVF (clusters + nprobe) → the sweep → PQ. Running it here ties the whole pipeline together in one labelled block.

In [ ]:
line = '=' * 70
print(line)
print('Part 4: Vector Databases and Indexing  (offline demo)')
print(f'point cloud: N={N} vectors, d={D} dims, seeded (reproducible)')
print(line)

print(f'\n[1] brute-force k-NN (exact, O(n)): top-{K} = {[int(i) for i in exact_top]}')
print(f'    {N} comparisons per query.')

best = max(rows, key=lambda r: r['speedup'] if r['recall'] >= 0.9 else -1)
print(f'\n[2] IVF index: {n_clusters} k-means shelves; nprobe slides the curve.')
print(f"    e.g. nprobe={best['nprobe']}: recall@{K}={best['recall']:.3f}, "
      f"{best['comparisons']:.0f} comparisons, {best['speedup']:.1f}x speedup.")

engine = 'FAISS IndexIVFFlat' if faiss_index is not None else 'transparent NumPy IVFIndex'
print(f'    engine in use: {engine}')

print(f'\n[3] PQ: each {D}-d vector -> {m} codes, {raw_bytes // pq_bytes}x smaller, '
      f'fidelity {fidelity:.3f}.')

print('\n' + line)
print('brute force is exact but O(n); IVF gives up a sliver of recall to skip')
print('most comparisons; nprobe slides you along that curve; PQ shrinks the')
print('vectors themselves. Speed vs recall, all the way down.')
print(line)

## What you learned

- **Brute-force k-NN** is exact but **O(n)** — one dot product per stored vector. Fine for thousands, unworkable at millions.
- Ordinary database indexes (the **B-tree**) cannot rescue us: nearest-neighbor in hundreds of dimensions is not a sorted-column lookup, and tree partitioning collapses under the **curse of dimensionality**.
- The fix is to stop being exact. **ANN** trades a little **recall** (the fraction of the true top-k you return) for a huge speed gain. Every index is a point on the **speed-vs-recall** curve.
- **IVF** clusters vectors around **centroids** (via **k-means**) and searches only the `nprobe` nearest shelves. We built it in plain NumPy and watched `nprobe` slide recall and speed in opposite directions.
- **HNSW** (the essay's other family) navigates a layered graph greedily — long hops up top, short hops at the bottom — for excellent speed and recall at a higher memory cost.
- **Product Quantization** compresses the vectors themselves into tiny codes, saving memory at a small accuracy cost; pair it with IVF as **IVF+PQ**.
- A **vector database** is *index + storage + API + operational machinery*, organized around similarity search — holding the vectors, their text and **metadata**, with **metadata filtering** and **upserts**. Start simple and local; graduate only when a real constraint demands it.

## Next

We can now retrieve the right chunks **fast at scale**. But we have leaned on one unexamined assumption since Part 1: that documents arrive pre-cut into tidy chunks. They do not. **Part 5 — Documents & Chunking** tackles the split: how to break real documents into pieces worth embedding, and why getting it wrong quietly sabotages everything downstream.

← Previous: **Part 3 — Measuring Similarity**  ·  Next: **Part 5 — Documents & Chunking** →